### Measures of Dispersion / Variability (Spread)
- While **central tendency** gives us the center of a dataset, 
- **measures of dispersion** describe how spread out or clustered the data points are around that center.

--- 
## 1. Range (Spread from Minimum to Maximum)

* **What it does:** Measures the total span between the largest and smallest observations.
* **Mathematical Notation:**
  $$R = x_{\max} - x_{\min}$$
  *Where:*
  * $R$ (*R*): Range
  * $x_{\max}$ (*x max*): Maximum value in the dataset
  * $x_{\min}$ (*x min*): Minimum value in the dataset
* **Naive Example:** Daily high temperatures over a week: `[18, 20, 22, 25, 30]` ($x_{\min} = 18, x_{\max} = 30$).
  $$R = 30 - 18 = 12^\circ\text{C}$$
* **When to use:** For quick, high-level checks where simple boundaries are needed.

In [1]:
import numpy as np

# 1. Range Example
temperatures = [18, 20, 22, 25, 30]

# Plain Python
range_python = max(temperatures) - min(temperatures)

# NumPy - Method A: Expressive Min/Max (Recommended in modern NumPy)
range_np_minmax = np.max(temperatures) - np.min(temperatures)

# NumPy - Method B: Peak-to-Peak function
range_np_ptp = np.ptp(temperatures)  # 'ptp' stands for peak-to-peak

print(f"Plain Python calculation:      {range_python}°C")
print(f"NumPy (np.max - np.min):        {range_np_minmax}°C")
print(f"NumPy (np.ptp):                {range_np_ptp}°C")

Plain Python calculation:      12°C
NumPy (np.max - np.min):        12°C
NumPy (np.ptp):                12°C


--- 
## 2. Interquartile Range (IQR - Middle 50% Spread)

* **What it does:** Measures the spread of the middle 50% of data, ignoring extreme top and bottom tail values.
* **Mathematical Notation:**
  $$\text{IQR} = Q_3 - Q_1$$
  *Where:*
  * $\text{IQR}$ (*I-Q-R*): Interquartile Range
  * $Q_1$ (*Q one*): 25th percentile (first quartile)
  * $Q_3$ (*Q three*): 75th percentile (third quartile)
* **Naive Example:** Exam scores: `[10, 50, 60, 70, 80, 85, 100]`. $Q_1 = 50$ and $Q_3 = 85$.
  $$\text{IQR} = 85 - 50 = 35 \text{ marks}$$
* **When to use:** When data has heavy outliers or skewness (e.g., house prices, incomes).

In [2]:
import math
import statistics
import numpy as np
import scipy.stats as stats

# 1. Core / Pure Python (From Scratch via Linear Interpolation)
def get_percentile_linear(data, percentile):
    """Calculates percentile using linear interpolation (NumPy default equivalent)."""
    sorted_data = sorted(data)
    n = len(sorted_data)
    
    # Fractional index position
    index = (percentile / 100) * (n - 1)
    lower_idx = math.floor(index)
    upper_idx = math.ceil(index)
    
    # Linear interpolation
    fraction = index - lower_idx
    return sorted_data[lower_idx] + fraction * (sorted_data[upper_idx] - sorted_data[lower_idx])

# 2. Weibull Method (N+1 Formula)
def get_percentile_weibull(data, percentile):
    """Calculates percentile using the Weibull formula: Rank = (p / 100) * (N + 1)."""
    sorted_data = sorted(data)
    n = len(sorted_data)
    
    # Weibull rank formula (1-based position conversion to 0-based index)
    rank = (percentile / 100) * (n + 1)
    index = rank - 1  # convert 1-based rank to 0-based index
    
    # Boundary guard if rank falls outside indices
    if index <= 0:
        return sorted_data[0]
    if index >= n - 1:
        return sorted_data[-1]
        
    lower_idx = math.floor(index)
    upper_idx = math.ceil(index)
    fraction = index - lower_idx
    return sorted_data[lower_idx] + fraction * (sorted_data[upper_idx] - sorted_data[lower_idx])


# Example dataset
scores = [10, 50, 60, 70, 80, 85, 100]

# using Core / Pure Python (From Scratch via Linear Interpolation)
q1_core = get_percentile_linear(scores, 25)
q3_core = get_percentile_linear(scores, 75)
iqr_core = q3_core - q1_core

# Plain Python (using statistics module for quantiles)
# using linear interpolation with the statistics module (method='inclusive')
q1_stats_linear, _, q3_stats_linear = statistics.quantiles(scores, n=4, method='inclusive')
iqr_python_linear = q3_stats_linear - q1_stats_linear

# NumPy Library
# using linear interpolation with the NumPy library
q1_np_linear, q3_np_linear = np.percentile(scores, [25, 75], method='linear')
iqr_np_linear = q3_np_linear - q1_np_linear

# SciPy Library
# stats.iqr defaults to linear interpolation
q1_sp = np.percentile(scores, 25, method='linear')
q3_sp = np.percentile(scores, 75, method='linear')
iqr_scipy = stats.iqr(scores)


# using Weibull Method (N+1 Formula / Excel QUARTILE.EXC / R Type 6)
q1_weibull = get_percentile_weibull(scores, 25)
q3_weibull = get_percentile_weibull(scores, 75)
iqr_weibull = q3_weibull - q1_weibull

# using weibull method with the statistics module (method='exclusive' is statistics default)
q1_stats_weibull, _, q3_stats_weibull = statistics.quantiles(scores, n=4, method='exclusive')
iqr_stats_weibull = q3_stats_weibull - q1_stats_weibull

# using Weibull method with the NumPy library
q1_np_weibull, q3_np_weibull = np.percentile(scores, [25, 75], method='weibull')
iqr_np_weibull = q3_np_weibull - q1_np_weibull

# using Weibull method with the SciPy library
q1_sp_weibull = np.percentile(scores, 25, method='weibull')
q3_sp_weibull = np.percentile(scores, 75, method='weibull')
iqr_scipy_weibull = stats.iqr(scores, interpolation='weibull')

# =========================================================
# OUTPUT DISPLAY
# =========================================================

print("--- 1. LINEAR INTERPOLATION (N - 1 / INCLUSIVE) ---")
print(f"Core Python linear interpolation (From Scratch):         {iqr_core:.1f}  (Q1={q1_core:.1f}, Q3={q3_core:.1f})")
print(f"Plain Python linear interpolation (statistics.quantiles): {iqr_python_linear:.1f}  (Q1={q1_stats_linear:.1f}, Q3={q3_stats_linear:.1f})")
print(f"NumPy linear interpolation (np.percentile):              {iqr_np_linear:.1f}  (Q1={q1_np_linear:.1f}, Q3={q3_np_linear:.1f})")
print(f"SciPy linear interpolation (stats.iqr):                  {iqr_scipy:.1f}  (Q1={q1_sp:.1f}, Q3={q3_sp:.1f})")

print("\n--- 2. WEIBULL METHOD (N + 1 / EXCLUSIVE) ---")
print(f"Core Python Weibull Method:                             {iqr_weibull:.1f}  (Q1={q1_weibull:.1f}, Q3={q3_weibull:.1f})")
print(f"Plain Python Weibull Method (statistics.quantiles):    {iqr_stats_weibull:.1f}  (Q1={q1_stats_weibull:.1f}, Q3={q3_stats_weibull:.1f})")
print(f"NumPy Weibull Method (np.percentile):                  {iqr_np_weibull:.1f}  (Q1={q1_np_weibull:.1f}, Q3={q3_np_weibull:.1f})")
print(f"SciPy Weibull Method (stats.iqr):                        {iqr_scipy_weibull:.1f}  (Q1={q1_sp_weibull:.1f}, Q3={q3_sp_weibull:.1f})")

--- 1. LINEAR INTERPOLATION (N - 1 / INCLUSIVE) ---
Core Python linear interpolation (From Scratch):         27.5  (Q1=55.0, Q3=82.5)
Plain Python linear interpolation (statistics.quantiles): 27.5  (Q1=55.0, Q3=82.5)
NumPy linear interpolation (np.percentile):              27.5  (Q1=55.0, Q3=82.5)
SciPy linear interpolation (stats.iqr):                  27.5  (Q1=55.0, Q3=82.5)

--- 2. WEIBULL METHOD (N + 1 / EXCLUSIVE) ---
Core Python Weibull Method:                             35.0  (Q1=50.0, Q3=85.0)
Plain Python Weibull Method (statistics.quantiles):    35.0  (Q1=50.0, Q3=85.0)
NumPy Weibull Method (np.percentile):                  35.0  (Q1=50.0, Q3=85.0)
SciPy Weibull Method (stats.iqr):                        35.0  (Q1=50.0, Q3=85.0)


## 3. Sample Variance (Average Squared Deviations)

* **What it does:** Measures how far points deviate from the mean on average, squared so positive and negative differences do not cancel out.
* **Mathematical Notation:**
  $$s^2 = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2$$
  *Where:*
  * $s^2$ (*s squared*): Sample variance
  * $n-1$ (*n minus one*): Bessel's correction for sample bias
  * $x_i$ (*x sub i*): i-th observation
  * $\bar{x}$ (*x-bar*): Sample mean
  * $\sum$ (*capital Sigma*): Summation operator
* **Naive Example:** Heights of 3 plants in cm: `[10, 20, 30]` ($n = 3$, $\bar{x} = 20$). Deviations are $(-10, 0, 10)$, squared deviations are $(100, 0, 100)$.
  $$s^2 = \frac{100 + 0 + 100}{3 - 1} = \frac{200}{2} = 100 \text{ cm}^2$$
* **When to use:** Foundation for inferential statistics and risk modeling (units are squared).

In [3]:
import statistics
import numpy as np

# 3. Sample Variance Example
plants = [10, 20, 30]

# Plain Python
mean_val = sum(plants) / len(plants)
var_python = sum((x - mean_val) ** 2 for x in plants) / (len(plants) - 1)

# Standard Library (statistics)
var_stats = statistics.variance(plants)

# NumPy Library (ddof=1 specifies sample variance with Bessel's correction)
var_np = np.var(plants, ddof=1) # ddof means "delta degrees of freedom", used to specify sample variance with Bessel's correction

print(f"Plain Python calculation:  {var_python:.1f} cm²")
print(f"statistics.variance():      {var_stats:.1f} cm²")
print(f"np.var(ddof=1):            {var_np:.1f} cm²")

Plain Python calculation:  100.0 cm²
statistics.variance():      100.0 cm²
np.var(ddof=1):            100.0 cm²


--- 
#### 4. Sample Standard Deviation (Spread in Original Units)

* **What it does:** Converts variance back into original measurement units by taking the square root.
* **Mathematical Notation:**
  $$s = \sqrt{s^2} = \sqrt{\frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$
  *Where:*
  * $s$ (*s*): Sample standard deviation
  * $s^2$ (*s squared*): Sample variance
  * $\sqrt{\cdot}$ (*square root*): Square root operator
* **Naive Example:** Using plant heights `[10, 20, 30]` with sample variance $s^2 = 100\text{ cm}^2$:
  $$s = \sqrt{100} = 10\text{ cm}$$
* **When to use:** Primary metric for describing spread in standard units alongside the mean.

In [ ]:
import math
import statistics
import numpy as np

# 4. Sample Standard Deviation Example
plants = [10, 20, 30]

# Plain Python
mean_val = sum(plants) / len(plants)
var_val = sum((x - mean_val) ** 2 for x in plants) / (len(plants) - 1)
std_python = math.sqrt(var_val)

# Standard Library (statistics)
std_stats = statistics.stdev(plants)

# NumPy Library (ddof=1 specifies sample standard deviation)
std_np = np.std(plants, ddof=1) # delta degree of freedom

print(f"Plain Python calculation:  {std_python:.1f} cm")
print(f"statistics.stdev():        {std_stats:.1f} cm")
print(f"np.std(ddof=1):            {std_np:.1f} cm")

Plain Python calculation:  10.0 cm
statistics.stdev():        10.0 cm
np.std(ddof=1):            10.0 cm


| Metric | Mathematical Formula | Plain Python | Standard Library (`statistics`) | NumPy |
| :--- | :--- | :--- | :--- | :--- |
| **Sample Variance** ($s^2$) | $\frac{1}{n-1} \sum (x_i - \bar{x})^2$ | `/ (n - 1)` | `statistics.variance()` | `np.var(ddof=1)` |
| **Population Variance** ($\sigma^2$) | $\frac{1}{N} \sum (x_i - \mu)^2$ | `/ N` | `statistics.pvariance()` | `np.var(ddof=0)` *(default)* |
| **Sample Std Dev** ($s$) | $\sqrt{\frac{1}{n-1} \sum (x_i - \bar{x})^2}$ | `math.sqrt(...)` | `statistics.stdev()` | `np.std(ddof=1)` |
| **Population Std Dev** ($\sigma$) | $\sqrt{\frac{1}{N} \sum (x_i - \mu)^2}$ | `math.sqrt(...)` | `statistics.pstdev()` | `np.std(ddof=0)` *(default)* |

--- 
## 5. Mean Absolute Deviation (MAD)

* **What it does:** Calculates the average absolute distance of all data points from the mean without squaring.
* **Mathematical Notation:**
  $$\text{MAD} = \frac{1}{n} \sum_{i=1}^{n} \vert{}x_i - \bar{x}\vert{}$$
  *Where:*
  * $\text{MAD}$ (*M-A-D*): Mean Absolute Deviation
  * $\vert{}\cdot\vert{}$ (*absolute value bars*): Absolute value operator (discards negative signs)
  * $n$ (*n*): Total observations
  * $\bar{x}$ (*x-bar*): Sample mean
* **Naive Example:** 3 daily deliveries in minutes: `[10, 20, 30]` ($\bar{x} = 20$). Absolute deviations are $\vert{}10-20\vert{} = 10, \vert{}20-20\vert{} = 0, \vert{}30-20\vert{} = 10$.
  $$\text{MAD} = \frac{10 + 0 + 10}{3} = \frac{20}{3} \approx 6.67 \text{ minutes}$$
* **When to use:** Highly intuitive spread metric that is less sensitive to extreme outliers than standard deviation.

In [5]:
import numpy as np

# 5. Mean Absolute Deviation Example
deliveries = [10, 20, 30]

# Plain Python
mean_val = sum(deliveries) / len(deliveries)
mad_python = sum(abs(x - mean_val) for x in deliveries) / len(deliveries)

# NumPy Library
mad_np = np.mean(np.abs(np.array(deliveries) - np.mean(deliveries)))

print(f"Plain Python calculation: {mad_python:.2f} minutes")
print(f"NumPy vector operations:   {mad_np:.2f} minutes")

Plain Python calculation: 6.67 minutes
NumPy vector operations:   6.67 minutes


--- 
## 6. Median Absolute Deviation (Median MAD)

* **What it does:** Measures robust dispersion by computing the median of absolute differences from the data's median.
* **Mathematical Notation:**
  $$\text{MAD}_{\text{med}} = \text{median}\left( \vert{}x_i - \tilde{x}\vert{} \right)$$
  *Where:*
  * $\text{MAD}_{\text{med}}$ (*median MAD*): Median Absolute Deviation
  * $\tilde{x}$ (*x-tilde*): Median of the dataset
  * $\vert{}x_i - \tilde{x}\vert{}$ (*absolute difference*): Distance of each point from the median
* **Naive Example:** Hourly wages: `[10, 12, 15, 20, 100]` ($\tilde{x} = 15$). Deviations from median: $\vert{}10-15\vert{}=5, \vert{}12-15\vert{}=3, \vert{}15-15\vert{}=0, \vert{}20-15\vert{}=5, \vert{}100-15\vert{}=85$. Sorted deviations: `[0, 3, 5, 5, 85]`.
  $$\text{MAD}_{\text{med}} = 5$$
* **When to use:** Extreme outlier situations where standard deviation and mean absolute deviation fail.

In [6]:
import statistics
import numpy as np
import scipy.stats as stats

# 6. Median Absolute Deviation Example
wages = [10, 12, 15, 20, 100]

# Plain Python
med_val = statistics.median(wages)
devs = [abs(x - med_val) for x in wages]
mad_med_python = statistics.median(devs)

# NumPy Library
mad_med_np = np.median(np.abs(np.array(wages) - np.median(wages)))

# SciPy Library
mad_med_scipy = stats.median_abs_deviation(wages, scale=1)

print(f"Plain Python calculation: ${mad_med_python:.2f}")
print(f"NumPy calculation:        ${mad_med_np:.2f}")
print(f"scipy.stats.median_abs_deviation(): ${mad_med_scipy:.2f}")

Plain Python calculation: $5.00
NumPy calculation:        $5.00
scipy.stats.median_abs_deviation(): $5.00


--- 
## 7. Coefficient of Variation (Relative Variability)

* **What it does:** Measures relative spread as a percentage ratio of standard deviation to mean, allowing comparisons across different units or scales.
* **Mathematical Notation:**
  $$\text{CV} = \frac{s}{\bar{x}} \times 100\%$$
  *Where:*
  * $\text{CV}$ (*C-V*): Coefficient of Variation
  * $s$ (*s*): Sample standard deviation
  * $\bar{x}$ (*x-bar*): Sample mean
* **Naive Example:** 
- Stocks A and B. Stock A ($\bar{x} = \$10, s = \$2 \implies \text{CV} = 20\%$) vs 
- Stock B ($\bar{x} = \$100, s = \$5 \implies \text{CV} = 5\%$). 
- Stock A has higher relative risk.
* **When to use:** Comparing variability between datasets with different measurement scales (e.g., comparing income variability in USD vs Yen).

In [7]:
import statistics
import scipy.stats as stats
import numpy as np

# 7. Coefficient of Variation Example
prices = [10, 12, 15, 18, 20]

# Plain Python
mean_val = sum(prices) / len(prices)
std_val = statistics.stdev(prices)
cv_python = (std_val / mean_val) * 100

# SciPy Library (stats.variation returns proportion; multiply by 100 for %)
cv_scipy_1 = stats.variation(prices) * 100 # uses population standard deviation(N) (ddof=0) by default
cv_scipy_2 = stats.variation(prices,ddof=1) * 100 # uses sample standard deviation(N-1) (ddof=1)

print(f"Plain Python calculation: {cv_python:.2f}%")
print(f"scipy.stats.variation() (population, ddof=0):  {cv_scipy_1:.2f}%")
print(f"scipy.stats.variation() (sample, ddof=1):      {cv_scipy_2:.2f}%")

Plain Python calculation: 27.49%
scipy.stats.variation() (population, ddof=0):  24.59%
scipy.stats.variation() (sample, ddof=1):      27.49%


### previous cell explained
### 1. Plain Python Calculation (Sample Std Dev, $n - 1 = 4$)

`statistics.stdev()` calculates sample standard deviation:

$$s = \sqrt{\frac{(10-15)^2 + (12-15)^2 + (15-15)^2 + (18-15)^2 + (20-15)^2}{5 - 1}} = \sqrt{\frac{68}{4}} = \mathbf{4.1231}$$

$$\text{CV}_{\text{python}} = \left(\frac{4.1231}{15.0}\right) \times 100\% = \mathbf{27.49\%}$$

---

### 2. SciPy Calculation (Population Std Dev, $N = 5$)

By default, `stats.variation()` divides by $N$ (`ddof=0`):

$$\sigma = \sqrt{\frac{68}{5}} = \mathbf{3.6878}$$

$$\text{CV}_{\text{scipy}} = \left(\frac{3.6878}{15.0}\right) \times 100\% = \mathbf{24.59\%}$$